In [9]:
import torch
import numpy as np
import random

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)  # for Apple GPU reproducibility

set_seed()

In [10]:
from torchvision import datasets, transforms
from torch.utils.data import random_split

transform = transforms.ToTensor()

full_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# MNIST already gives us a separate test set (10,000 images).
# We still need to carve validation out of the 60,000 training images.
val_size = 6000                      # 10% of 60,000
train_size = len(full_train) - val_size

train_data, val_data = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)  # controlled randomness for the split itself
)

print("Train:", len(train_data))
print("Val:  ", len(val_data))
print("Test: ", len(test_data))

Train: 54000
Val:   6000
Test:  10000


In [11]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1 input channel (grayscale) -> 16 feature maps
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 -> 32 feature maps
        self.pool  = nn.MaxPool2d(2)                              # halves spatial size each time
        self.fc1   = nn.Linear(32 * 7 * 7, 128)                   # flattened features -> 128
        self.fc2   = nn.Linear(128, 10)                           # 128 -> 10 digit classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.view(x.size(0), -1)              # flatten to a vector per image
        x = F.relu(self.fc1(x))
        x = self.fc2(x)                        # raw scores per class (no softmax needed here)
        return x

In [12]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [13]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

In [14]:
import torch.optim as optim

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()  # tells layers like dropout/batchnorm "we're training" (we don't have any here, but it's good habit)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear old gradients
        outputs = model(images)        # forward pass: get predictions
        loss = criterion(outputs, labels)  # how wrong were we
        loss.backward()                # compute gradients (backprop)
        optimizer.step()               # update weights using those gradients

In [15]:
def evaluate(model, loader, device):
    model.eval()  # tells layers "we're evaluating now" (again, matters more with dropout/batchnorm)
    correct = 0
    total = 0
    with torch.no_grad():  # don't track gradients — we're not training here
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)   # pick the class with the highest score
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total   # accuracy as a fraction, e.g. 0.97

In [16]:
NUM_EPOCHS = 5

model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(NUM_EPOCHS):
    train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_acc = evaluate(model, val_loader, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Validation accuracy: {val_acc:.4f}")

Epoch 1/5 - Validation accuracy: 0.9758
Epoch 2/5 - Validation accuracy: 0.9813
Epoch 3/5 - Validation accuracy: 0.9858
Epoch 4/5 - Validation accuracy: 0.9858
Epoch 5/5 - Validation accuracy: 0.9868


In [17]:
test_acc = evaluate(model, test_loader, device)
print(f"Final test accuracy: {test_acc:.4f}")

Final test accuracy: 0.9893
